<a href="https://colab.research.google.com/github/24dhanush/GUVI_PROJECT/blob/project-4/Bitcoinapp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install streamlit

In [ ]:
%%writefile bitcoinapp.py
import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

import tensorflow as tf
from tensorflow.keras import layers, models

# --------------------------------------------------
# PAGE CONFIG
# --------------------------------------------------
st.set_page_config(
    page_title="Bitcoin Prediction",
    layout="wide"
)

# --------------------------------------------------
# TITLE
# --------------------------------------------------
st.title("📈 Project 4 - Bitcoin Price Prediction")

st.markdown("---")

# --------------------------------------------------
# SIDEBAR
# --------------------------------------------------
st.sidebar.header("Model Selection")

selected_model = st.sidebar.selectbox(
    "Choose Prediction Model",
    [
        "LSTM",
        "RNN",
        "CNN",
        "Transformer"
    ]
)

epochs = st.sidebar.slider(
    "Epochs",
    5,
    50,
    10
)

batch_size = st.sidebar.selectbox(
    "Batch Size",
    [16,32,64],
    index=1
)

uploaded_file = st.sidebar.file_uploader(
    "/content/Bitcoin Historical Data (1).csv",
    type=["csv"]
)

run = st.sidebar.button("Train Model")

# --------------------------------------------------
# FUNCTIONS
# --------------------------------------------------

def create_sequences(data, window_size=60, horizon=3):

    X=[]
    y=[]

    for i in range(window_size,len(data)-horizon+1):
        X.append(data[i-window_size:i])
        y.append(data[i:i+horizon])

    return np.array(X),np.array(y)


def build_lstm(input_shape,horizon):

    model=models.Sequential([
        layers.LSTM(64,input_shape=input_shape),
        layers.Dense(32,activation="relu"),
        layers.Dense(horizon)
    ])

    model.compile(
        optimizer="adam",
        loss="mse"
    )

    return model


def build_rnn(input_shape,horizon):

    model=models.Sequential([
        layers.SimpleRNN(50,input_shape=input_shape),
        layers.Dense(horizon)
    ])

    model.compile(
        optimizer="adam",
        loss="mse"
    )

    return model


def build_cnn(input_shape,horizon):

    model=models.Sequential([

        layers.Conv1D(
            filters=64,
            kernel_size=3,
            activation="relu",
            input_shape=input_shape
        ),

        layers.MaxPooling1D(2),

        layers.Flatten(),

        layers.Dense(50,activation="relu"),

        layers.Dense(horizon)

    ])

    model.compile(
        optimizer="adam",
        loss="mse"
    )

    return model


def build_transformer(input_shape,horizon):

    inputs=layers.Input(shape=input_shape)

    attention=layers.MultiHeadAttention(
        num_heads=4,
        key_dim=input_shape[-1]
    )(inputs,inputs)

    x=layers.LayerNormalization()(attention+inputs)

    x=layers.GlobalAveragePooling1D()(x)

    x=layers.Dense(
        64,
        activation="relu"
    )(x)

    outputs=layers.Dense(horizon)(x)

    model=models.Model(inputs,outputs)

    model.compile(
        optimizer="adam",
        loss="mse"
    )

    return model


# --------------------------------------------------
# TRAIN
# --------------------------------------------------

if uploaded_file is not None:

    df=pd.read_csv(uploaded_file)

    st.subheader("Dataset Preview")

    st.dataframe(df.head())

    df["Price"]=df["Price"].str.replace(",","").astype(float)

    data=df[["Price"]].values

    scaler=MinMaxScaler()

    scaled_data=scaler.fit_transform(data)

    X,y=create_sequences(
        scaled_data,
        window_size=60,
        horizon=3
    )

    split=int(len(X)*0.8)

    X_train=X[:split]
    X_test=X[split:]

    y_train=y[:split]
    y_test=y[split:]

    if run:

        st.info("Training Model...")

        if selected_model=="LSTM":

            model=build_lstm((60,1),3)

        elif selected_model=="RNN":

            model=build_rnn((60,1),3)

        elif selected_model=="CNN":

            model=build_cnn((60,1),3)

        else:

            model=build_transformer((60,1),3)

        history=model.fit(

            X_train,
            y_train,

            validation_data=(X_test,y_test),

            epochs=epochs,

            batch_size=batch_size,

            verbose=0

        )

        pred=model.predict(X_test)

        pred_actual=scaler.inverse_transform(
            pred.reshape(-1,1)
        )

        y_actual=scaler.inverse_transform(
            y_test.reshape(-1,1)
        )

        mae=mean_absolute_error(
            y_actual,
            pred_actual
        )

        rmse=np.sqrt(
            mean_squared_error(
                y_actual,
                pred_actual
            )
        )

        mape=mean_absolute_percentage_error(
            y_actual,
            pred_actual
        )*100

        st.success("Training Completed")

        # -----------------------------
        # METRICS
        # -----------------------------

        st.subheader("Evaluation Metrics")

        c1,c2,c3=st.columns(3)

        c1.metric("MAE",f"{mae:.2f}")

        c2.metric("RMSE",f"{rmse:.2f}")

        c3.metric("MAPE",f"{mape:.2f}%")

        # -----------------------------
        # ACTUAL VS PREDICTED
        # -----------------------------

        st.subheader("Actual vs Predicted")

        fig,ax=plt.subplots(figsize=(12,5))

        ax.plot(
            y_actual,
            label="Actual",
            color="blue"
        )

        ax.plot(
            pred_actual,
            label="Predicted",
            color="red"
        )

        ax.legend()

        st.pyplot(fig)

        # -----------------------------
        # LOSS CURVE
        # -----------------------------

        st.subheader("Training Loss")

        fig2,ax2=plt.subplots(figsize=(12,5))

        ax2.plot(
            history.history["loss"],
            label="Train Loss"
        )

        ax2.plot(
            history.history["val_loss"],
            label="Validation Loss"
        )

        ax2.legend()

        st.pyplot(fig2)

        # -----------------------------
        # FUTURE PREDICTION
        # -----------------------------

        st.subheader("Next 3-Day Prediction")

        last60=scaled_data[-60:]

        future=model.predict(
            last60.reshape(1,60,1)
        )

        future=scaler.inverse_transform(
            future.reshape(-1,1)
        )

        future=pd.DataFrame({

            "Day":[
                "Day 1",
                "Day 2",
                "Day 3"
            ],

            "Predicted Price $":future.flatten()

        })

        st.table(future)
# -----------------------------
# Footer
# -----------------------------

st.sidebar.markdown("---")
st.sidebar.write("Created by Dhanush DB")


In [ ]:
!streamlit run bitcoinapp.py & npx localtunnel --port 8501